# Stable Diffusion 1.5 + LoRA for longitudinal face aging

This notebook loads exactly one SD1.5-compatible backbone. It freezes the VAE, CLIP text encoder, and base U-Net; expands `conv_in` from 4 to 8 latent channels; and trains LoRA, `conv_in`, plus a small relative-age MLP injected into the timestep embedding. Running the model-construction cell may download model weights unless `MODEL_ID` points to an existing local directory.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'model').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required = ('diffusers', 'transformers')
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(
        f'Missing server dependencies: {missing}. Install diffusers, transformers, '
        'accelerate and safetensors in the existing deep_learning environment.'
    )

from data import build_face_aging_dataloaders
from src.model import (
    DEFAULT_MODEL_ID,
    build_face_aging_diffusion_bundle,
    build_face_aging_optimizer,
    inspect_model_batch,
    prepare_face_aging_forward,
    run_face_aging_model_validation,
    tokenizer_audit,
)

The old `runwayml/stable-diffusion-v1-5` repository is deprecated. The maintained Diffusers mirror below contains the expected `tokenizer/`, `text_encoder/`, `unet/`, `vae/`, and `scheduler/` subfolders. A local model directory can be used instead of a Hub ID.

In [ ]:
MODEL_ID = DEFAULT_MODEL_ID  # 'stable-diffusion-v1-5/stable-diffusion-v1-5'
VAE_ID = None                # Internal SD1.5 VAE
# For a noVAE checkpoint, for example:
# MODEL_ID = 'SG161222/Realistic_Vision_V6.0_B1_noVAE'
# VAE_ID = 'stabilityai/sd-vae-ft-mse'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32
print('Device:', DEVICE, 'frozen-weight dtype:', DTYPE)

In [ ]:
bundle = build_face_aging_diffusion_bundle(
    model_id=MODEL_ID,
    vae_id=VAE_ID,
    adapter_type='lora',
    rank=16,
    alpha=16,
    dropout=0.0,
    target_modules=('to_q', 'to_k', 'to_v', 'to_out.0'),
    source_conditioning='concat',
    use_age_delta_conditioning=True,
    age_conditioning_mode='delta_mlp',
    age_delta_scale=80.0,
    age_condition_hidden_dim=128,
    age_condition_output_dim=None,  # inferred as 1280 for SD1.5
    device=DEVICE,
    dtype=DTYPE,
    trainable_dtype=torch.float32,
    local_files_only=False,  # Set True after the model is cached or for a local path
)

assert bundle['unet'].config.in_channels == 8
assert bundle['conv_in_report']['source_weight_max_abs'] == 0.0
assert all(not p.requires_grad for p in bundle['vae'].parameters())
assert all(not p.requires_grad for p in bundle['text_encoder'].parameters())

`conv_in`, LoRA, and the age-conditioner parameters remain FP32. The helper forward uses autocast when the frozen backbone is FP16/BF16.

In [ ]:
stats = bundle['param_stats']
print('UNet total parameters:', f"{stats['unet']['total_params']:,}")
print('UNet trainable parameters:', f"{stats['unet']['trainable_params']:,}")
print('UNet trainable percentage:', f"{stats['unet']['trainable_pct']:.6f}%")
print('Adapter targets:', bundle['adapter_report']['counts_by_target'])
print('Age conditioner:', bundle['age_conditioning_config'])
print('Age-conditioner parameters:', f"{stats['age_delta_conditioner']['trainable_params']:,}")
print('Trainable tensors:', len(bundle['trainable_param_names']))
for name in bundle['trainable_param_names']:
    print(' -', name)

In [ ]:
optimizer = build_face_aging_optimizer(
    bundle,
    lr_lora=3e-5,
    lr_conv_in=5e-6,
    lr_age_conditioner=1e-4,
    weight_decay=1e-2,
    conv_in_weight_decay=1e-2,
    age_conditioner_weight_decay=1e-2,
)
for group in optimizer.param_groups:
    print(group['group_name'], 'lr=', group['lr'], 'tensors=', len(group['params']))

## Integration with the longitudinal DataLoader

Change only `DATASET_ROOT` on the server. No training loss or training loop is implemented in this milestone.

In [ ]:
DATASET_ROOT = PROJECT_ROOT / 'data' / 'sample'
# DATASET_ROOT = Path('/server/path/dataset_unificado')

loaders, data_metadata = build_face_aging_dataloaders(
    DATASET_ROOT, image_size=256, batch_size=2, num_workers=0,
    train_drop_last=False, train_shuffle=False, seed=42,
)
batch = next(iter(loaders['train']))
prepared = prepare_face_aging_forward(
    bundle, batch['source_image'], batch['target_image'], batch['target_prompt'],
    delta_ages=batch['delta_age'],
)
inspect_model_batch(batch, prepared)
assert prepared['conditioned_input'].shape[1] == 8
assert prepared['noise_pred'].shape == prepared['noise'].shape

In [ ]:
validation_report = run_face_aging_model_validation(bundle, batch=batch)
print('PASSED:', validation_report['passed'])
print('ERRORS:', validation_report['errors'])
print('WARNINGS:', validation_report['warnings'])
print('GRADIENTS:', validation_report['gradient_tests'])

Optional tokenizer diagnostic. It intentionally does not assume that `52-year-old` is a single CLIP token.

In [ ]:
for age in (5, 12, 25, 52, 80):
    prompts = [
        f'photo of a person as {age}-year-old',
        f'photo of a {age} year old person',
    ]
    for item in tokenizer_audit(bundle, prompts):
        print(item)